# SIFT extraction

In [6]:
"""df['SIFT_Features'][0]"""
# Normal sift gave 647040 features, which was too much to handle. so we limit the number of features to 100.

array([28., 41., 17., ...,  0.,  0.,  0.], shape=(647040,), dtype=float32)

In [8]:
import os
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm

# Set the path to the main directory
main_dir = "/Users/kshitijsharma/Old Documents/College/Surbhi ma'am work/ADL/ADL"
output_file = "sift_compact_features.csv"

# Initialize SIFT detector with a limit on keypoints
sift = cv2.SIFT_create(nfeatures=100)

# Prepare lists to hold data
image_indices = []
class_labels = []
compact_features = []

# Count total images for progress bar
total_images = sum(len(files) for _, _, files in os.walk(main_dir))

# Traverse through each class directory with a progress bar
image_index = 0
with tqdm(total=total_images, desc="Extracting SIFT features") as pbar:
    for class_label in os.listdir(main_dir):
        class_path = os.path.join(main_dir, class_label)
        
        if os.path.isdir(class_path):
            for image_name in os.listdir(class_path):
                image_path = os.path.join(class_path, image_name)
                
                # Read image
                img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
                if img is None:
                    pbar.update(1)
                    continue

                # Extract SIFT features with limited keypoints
                keypoints, descriptors = sift.detectAndCompute(img, None)
                
                # Aggregate descriptors by averaging
                if descriptors is not None and len(descriptors) > 0:
                    averaged_features = np.mean(descriptors, axis=0)
                else:
                    averaged_features = np.zeros(128)  # Default to zeros if no descriptors

                # Store the data
                image_indices.append(image_index)
                class_labels.append(class_label)
                compact_features.append(averaged_features)
                
                image_index += 1
                pbar.update(1)

# Convert data to DataFrame
data_dict = {"Image_Index": image_indices, "Class_Label": class_labels, "SIFT_Features": compact_features}
df = pd.DataFrame(data_dict)

# Save DataFrame to CSV
df.to_csv(output_file, index=False)

print(f"Compact SIFT features extracted and saved to {output_file}")

df['SIFT_Features'][0]

Extracting SIFT features: 100%|██████████| 963/963 [01:53<00:00,  8.49it/s]


Compact SIFT features extracted and saved to sift_compact_features.csv


array([25.04, 32.03, 27.67, 21.03, 13.19, 11.08, 10.18, 12.53, 48.9 ,
       39.29, 20.43, 17.66, 21.09, 26.47, 21.76, 26.66, 29.08, 20.65,
       13.86, 22.69, 38.2 , 39.37, 30.11, 27.09, 19.85, 16.48, 15.09,
       19.71, 24.42, 23.35, 19.1 , 21.  , 33.43, 23.56, 23.56, 24.93,
       24.79, 20.96, 18.91, 18.42, 89.23, 48.6 , 35.48, 36.3 , 36.57,
       25.74, 20.94, 42.27, 69.92, 45.32, 23.02, 33.23, 71.86, 56.29,
       25.93, 29.29, 31.63, 16.87, 17.15, 26.88, 42.47, 35.59, 20.66,
       21.37, 27.79, 12.93, 15.3 , 24.04, 29.02, 27.8 , 21.69, 21.93,
       91.27, 43.49, 19.94, 28.54, 39.43, 36.05, 25.66, 43.86, 71.61,
       32.13, 26.95, 55.99, 72.12, 31.86, 18.29, 42.23, 31.61, 22.69,
       24.4 , 31.  , 42.74, 26.83, 20.1 , 18.74, 17.77, 11.98, 15.05,
       18.05, 17.86, 15.28, 21.83, 23.62, 52.98, 34.08, 22.14, 22.  ,
       23.94, 16.35, 18.6 , 37.94, 32.61, 31.61, 32.23, 39.16, 43.65,
       25.9 , 13.34, 20.02, 22.76, 25.25, 23.63, 21.06, 23.01, 20.03,
       16.74, 16.15]

# LBP

In [37]:
import os
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm
from skimage.feature import local_binary_pattern

# Function to calculate LBP features using skimage
def calculate_lbp_fast(image):
    return local_binary_pattern(image, P=8, R=1, method="uniform")

# Set the path to the main directory
main_dir = "/Users/kshitijsharma/Old Documents/College/Surbhi ma'am work/ADL/ADL"
output_file = "lbp_features.csv"

# Prepare lists to hold data
image_indices = []
class_labels = []
lbp_data = []

# Count total images for progress bar
total_images = sum(len(files) for _, _, files in os.walk(main_dir))

# Traverse directories without sorting (match SIFT behavior)
image_index = 0
with tqdm(total=total_images, desc="Extracting LBP features") as pbar:
    for class_label in os.listdir(main_dir):  # No sorting
        class_path = os.path.join(main_dir, class_label)
        
        if os.path.isdir(class_path):
            for image_name in os.listdir(class_path):  # No sorting
                image_path = os.path.join(class_path, image_name)
                
                # Read image
                img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
                if img is None:
                    pbar.update(1)
                    continue

                # Extract LBP features
                lbp_image = calculate_lbp_fast(img)
                lbp_hist = np.histogram(lbp_image.ravel(), bins=np.arange(62))[0]

                # Remove trailing zeros
                lbp_hist = lbp_hist[:np.max(np.nonzero(lbp_hist)) + 1]

                # Store the data
                image_indices.append(image_index)
                class_labels.append(class_label)
                lbp_data.append(lbp_hist.flatten().tolist())

                image_index += 1
                pbar.update(1)

# Convert data to DataFrame
data_dict = {"Image_Index": image_indices, "Class_Label": class_labels, "LBP_Features": lbp_data}
df = pd.DataFrame(data_dict)

# Save DataFrame to CSV
df.to_csv(output_file, index=False)

print(f"LBP features extracted and saved to {output_file}")

Extracting LBP features: 100%|██████████| 963/963 [01:00<00:00, 15.85it/s]

LBP features extracted and saved to lbp_features.csv


# GLCM

In [5]:
import os
import cv2
import numpy as np
import pandas as pd
from skimage.feature import graycomatrix, graycoprops
from joblib import Parallel, delayed
from tqdm import tqdm

# Set the path to the main directory
main_dir = "/Users/kshitijsharma/Old Documents/College/Surbhi ma'am work/ADL/ADL"
output_file = "glcm_features.csv"
distances = [1]
angles = [0, np.pi/4, np.pi/2, 3*np.pi/4]

# Define GLCM extraction function
def extract_glcm_features(image_path):
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None
    glcm = graycomatrix(img, distances=distances, angles=angles, symmetric=True, normed=True)
    features = [
        graycoprops(glcm, 'contrast').flatten(),
        graycoprops(glcm, 'dissimilarity').flatten(),
        graycoprops(glcm, 'homogeneity').flatten(),
        graycoprops(glcm, 'energy').flatten(),
        graycoprops(glcm, 'correlation').flatten(),
        graycoprops(glcm, 'ASM').flatten()
    ]
    return np.concatenate(features).tolist()

# Prepare lists to hold data
image_indices = []
class_labels = []
glcm_data = []

# Traverse through each class directory with progress bar
image_index = 0
all_images = []
for class_label in os.listdir(main_dir):
    class_path = os.path.join(main_dir, class_label)
    if os.path.isdir(class_path):
        for image_name in os.listdir(class_path):
            image_path = os.path.join(class_path, image_name)
            all_images.append((image_index, class_label, image_path))
            image_index += 1

# Parallel processing with progress bar
results = Parallel(n_jobs=-1)(delayed(extract_glcm_features)(img_path) for _, _, img_path in tqdm(all_images))

# Collect results
for idx, (image_index, class_label, _) in enumerate(all_images):
    image_indices.append(image_index)
    class_labels.append(class_label)
    glcm_data.append(results[idx] if results[idx] is not None else [])

# Convert data to DataFrame
data_dict = {"Image_Index": image_indices, "Class_Label": class_labels, "GLCM_Features": glcm_data}
df = pd.DataFrame(data_dict)


100%|██████████| 963/963 [00:03<00:00, 255.87it/s]


In [6]:
df.head()

,Image_Index,Class_Label,GLCM_Features
0,0,Kidney - normal,"[20.072122625782384, 41.49451935865162, 25.937..."
1,1,Kidney - normal,"[41.46319889969735, 85.81752870152998, 46.2042..."
2,2,Kidney - normal,"[33.986849820640884, 89.05693911269617, 63.494..."
3,3,Kidney - normal,"[29.3155252598427, 55.30670516314655, 29.19134..."
4,4,Kidney - normal,"[16.191182050221084, 29.134457154325172, 19.47..."


In [7]:
# Save DataFrame to CSV
df.to_csv(output_file, index=False)
print(f"GLCM features extracted and saved to {output_file}")

GLCM features extracted and saved to glcm_features.csv


# GLDM

In [8]:
import os
import cv2
import numpy as np
import pandas as pd
from skimage.feature import graycomatrix, graycoprops
from tqdm import tqdm
from joblib import Parallel, delayed

# Define GLDM feature extraction function
def extract_gldm_features(image_path):
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return []
    
    # Compute Gray Level Dependency Matrix (GLDM)
    glcm = graycomatrix(img, [1], [0, np.pi/4, np.pi/2, 3*np.pi/4], 256, symmetric=True, normed=True)
    contrast = graycoprops(glcm, 'contrast').flatten().tolist()
    dissimilarity = graycoprops(glcm, 'dissimilarity').flatten().tolist()
    homogeneity = graycoprops(glcm, 'homogeneity').flatten().tolist()
    energy = graycoprops(glcm, 'energy').flatten().tolist()
    correlation = graycoprops(glcm, 'correlation').flatten().tolist()
    ASM = graycoprops(glcm, 'ASM').flatten().tolist()
    
    return contrast + dissimilarity + homogeneity + energy + correlation + ASM

# Set the path to the main directory
main_dir = "/Users/kshitijsharma/Old Documents/College/Surbhi ma'am work/ADL/ADL"
output_file = "gldm_features.csv"

# Prepare lists to hold data
image_indices = []
class_labels = []
gldm_data = []

# Traverse through directories and collect image paths
image_paths = []
class_names = []

for class_label in os.listdir(main_dir):
    class_path = os.path.join(main_dir, class_label)
    if os.path.isdir(class_path):
        for image_name in os.listdir(class_path):
            image_paths.append(os.path.join(class_path, image_name))
            class_names.append(class_label)

# Process images in parallel with a progress bar
results = Parallel(n_jobs=-1)(delayed(extract_gldm_features)(path) for path in tqdm(image_paths))

# Build DataFrame with matching image indices
for idx, (class_label, features) in enumerate(zip(class_names, results)):
    image_indices.append(idx)
    class_labels.append(class_label)
    gldm_data.append(features)

# Convert data to DataFrame
data_dict = {"Image_Index": image_indices, "Class_Label": class_labels, "GLDM_Features": gldm_data}
df = pd.DataFrame(data_dict)

100%|██████████| 963/963 [00:03<00:00, 244.63it/s]


In [9]:
df.head()

,Image_Index,Class_Label,GLDM_Features
0,0,Kidney - normal,"[20.072122625782384, 41.49451935865162, 25.937..."
1,1,Kidney - normal,"[41.46319889969735, 85.81752870152998, 46.2042..."
2,2,Kidney - normal,"[33.986849820640884, 89.05693911269617, 63.494..."
3,3,Kidney - normal,"[29.3155252598427, 55.30670516314655, 29.19134..."
4,4,Kidney - normal,"[16.191182050221084, 29.134457154325172, 19.47..."


In [10]:
# Save DataFrame to CSV
df.to_csv(output_file, index=False)
print(f"GLDM features extracted and saved to {output_file}")

GLDM features extracted and saved to gldm_features.csv


# GLRLM

In [12]:
import os
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm
from joblib import Parallel, delayed
import mahotas as mh

# Set the path to the main directory
main_dir = "/Users/kshitijsharma/Old Documents/College/Surbhi ma'am work/ADL/ADL"
output_file = "glrlm_features.csv"

# Function to extract GLRLM features from an image
def extract_glrlm_features(image_path):
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None
    
    # Compute GLRLM features using Mahotas
    glrlm_features = mh.features.haralick(img).mean(axis=0)
    return glrlm_features.tolist()

# Collect image paths and class labels
image_paths = []
class_labels = []

for class_label in os.listdir(main_dir):
    class_path = os.path.join(main_dir, class_label)
    if os.path.isdir(class_path):
        for image_name in os.listdir(class_path):
            image_paths.append(os.path.join(class_path, image_name))
            class_labels.append(class_label)

# Use parallel processing to extract features with a progress bar
results = Parallel(n_jobs=-1)(delayed(extract_glrlm_features)(path) for path in tqdm(image_paths, desc="Processing Images"))

# Assign indices
image_indices = list(range(len(image_paths)))

# Prepare DataFrame
data_dict = {"Image_Index": image_indices, "Class_Label": class_labels, "GLRLM_Features": results}
df = pd.DataFrame(data_dict)


Processing Images: 100%|██████████| 963/963 [00:03<00:00, 287.59it/s]


In [ ]:
df['GLRLM_Features'][1]

[0.0008080107119523462,
 62.51090006864917,
 0.9481647114243186,
 602.9686669631055,
 0.2243593502114171,
 296.0422847494938,
 2349.3637677837614,
 7.555630798573839,
 11.272222003929125,
 0.0004228404604585851,
 3.8803851067517146,
 -0.28496541633256106,
 0.9869301410611137]

In [16]:
# Save DataFrame to CSV
df.to_csv(output_file, index=False)
print(f"GLRLM features extracted and saved to {output_file}")

GLRLM features extracted and saved to glrlm_features.csv


# Combine features

In [53]:
import os
import pandas as pd

# Set path to folder containing CSV files
folder_path = "/Users/kshitijsharma/Old Documents/College/Surbhi ma'am work/Final work/Feature Extraction"

# Load all CSV files
csv_files = [f for f in os.listdir(folder_path) if f.endswith('.csv')]
dataframes = [pd.read_csv(os.path.join(folder_path, f)) for f in csv_files]

# Check if all dataframes have the same indexes and labels
reference_df = dataframes[0][['Image_Index', 'Class_Label']]
all_match = True

for i, df in enumerate(dataframes[1:], start=2):
    if not df[['Image_Index', 'Class_Label']].equals(reference_df):
        print(f"Mismatch found in file {csv_files[i-1]}")
        mismatched_rows = df[['Image_Index', 'Class_Label']].compare(reference_df)
        print("Mismatched rows:\n", mismatched_rows)
        all_match = False

if all_match:
    print("✅ All CSV files have matching indexes and labels!")
else:
    print("❌ Mismatches detected. Check the files listed above.")

✅ All CSV files have matching indexes and labels!


In [54]:
import pandas as pd
import ast
import numpy as np

# Define file paths
file_paths = [
    "glcm_features.csv",
    "gldm_features.csv",
    "glrlm_features.csv",
    "lbp_features.csv",
    "sift_compact_features.csv"
]

# Initialize lists to store fixed features
glcm = []
gldm = []
glrlm = []
lbp = []
sift = []

# Fix GLCM, GLDM, GLRLM, LBP (stringified lists)
for i, (file, feature_list) in enumerate(zip(file_paths[:4], [glcm, gldm, glrlm, lbp])):
    df = pd.read_csv(file)
    for feature_str in df.iloc[:, -1]:  # Assume last column is the feature
        try:
            feature_list.append(ast.literal_eval(feature_str))
        except Exception as e:
            print(f"Error parsing features from {file}: {e}")

# Fix SIFT (stringified numpy array with line breaks)
try:
    df_sift = pd.read_csv(file_paths[4])
    for feature_str in df_sift.iloc[:, -1]:
        try:
            # Remove line breaks, excessive spaces, and non-numeric parts
            clean_str = ''.join(c if c.isdigit() or c in ".- " else ' ' for c in feature_str)
            clean_str = " ".join(clean_str.split())  # Ensure only single spaces
            sift_array = np.array([float(x) for x in clean_str.split()])
            sift.append(sift_array.tolist())
        except Exception as e:
            print(f"Error parsing SIFT features: {e}")


except Exception as e:
    print(f"Error loading SIFT file: {e}")

# Output to verify
print(f"GLCM features parsed: {len(glcm)} entries")
print(f"GLDM features parsed: {len(gldm)} entries")
print(f"GLRLM features parsed: {len(glrlm)} entries")
print(f"LBP features parsed: {len(lbp)} entries")
print(f"SIFT features parsed: {len(sift)} entries")

GLCM features parsed: 963 entries
GLDM features parsed: 963 entries
GLRLM features parsed: 963 entries
LBP features parsed: 963 entries
SIFT features parsed: 963 entries


In [61]:
# Combine all features index-wise
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd

combined_features = []

# Ensure all lists have the same length
assert len(glcm) == len(gldm) == len(glrlm) == len(lbp) == len(sift), "Feature lists are not aligned!"

# Concatenate features for each image index
for i in range(len(glcm)):
    combined = glcm[i] + gldm[i] + glrlm[i] + lbp[i] + sift[i]
    combined_features.append(combined)

print(f"Combined features created for {len(combined_features)} images!")

# Standardize the combined features
scaler = StandardScaler()
combined_features = scaler.fit_transform(np.array(combined_features))

print("Combined features standardized!")

# Load one CSV to get image indexes and class labels (assuming all files share the same index/label structure)
reference_df = pd.read_csv("glcm_features.csv")

# Create a new DataFrame with image indexes, labels, and combined features
final_df = pd.DataFrame({
    'Image_Index': reference_df['Image_Index'],
    'Class_Label': reference_df['Class_Label'],
    'Combined_Features': [list(row) for row in combined_features]
})

# Save to a new CSV
final_df.to_csv("combined_features.csv", index=False)

print("Combined features saved to 'combined_features.csv'!")

Combined features created for 963 images!
Combined features standardized!
Combined features saved to 'combined_features.csv'!
